In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, make_scorer
import warnings
warnings.filterwarnings('ignore')

# Load cleaned data
data_path = r"C:\Users\DELL G3\Desktop\RMIT SEM - 4\Loan Club Project\outputs\loans_cleaned.parquet"
df = pd.read_parquet(data_path)

# Remove circular predictors
X = df.drop(columns=['target', 'grade', 'int_rate'])
y = df['target']

# Same split as before - same random_state guarantees identical split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print(f"Training set: {X_train.shape[0]:,} rows")
print(f"Test set:     {X_test.shape[0]:,} rows")
print(f"Features:     {X_train.shape[1]}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

Training set: 1,075,552 rows
Test set:     268,889 rows
Features:     76
scale_pos_weight: 4.01


In [3]:
# The parameters we want to tune and the ranges to search
param_distributions = {
    # Number of trees — more trees = more learning but slower
    'n_estimators': [200, 300, 400, 500],
    
    # How deep each tree grows — deeper = more complex patterns
    # but risks overfitting
    'max_depth': [3, 4, 5, 6, 7, 8],
    
    # How much each tree contributes — lower = more careful learning
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    
    # Fraction of training rows used per tree — adds randomness
    # prevents overfitting
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    
    # Fraction of features used per tree — also prevents overfitting
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    
    # Minimum samples required in a leaf node
    # Higher = more conservative, less overfitting
    'min_child_weight': [1, 3, 5, 7],
    
    # L2 regularisation — penalises model complexity
    'reg_lambda': [0.1, 1.0, 5.0, 10.0],
}

print("Parameter search space:")
total_combinations = 1
for param, values in param_distributions.items():
    print(f"  {param:<25} {values}")
    total_combinations *= len(values)

print(f"\nTotal possible combinations: {total_combinations:,}")
print(f"We will randomly sample:     20")
print(f"Time saved:                  ~{total_combinations/20:.0f}x faster than GridSearch")

Parameter search space:
  n_estimators              [200, 300, 400, 500]
  max_depth                 [3, 4, 5, 6, 7, 8]
  learning_rate             [0.01, 0.03, 0.05, 0.1]
  subsample                 [0.6, 0.7, 0.8, 0.9, 1.0]
  colsample_bytree          [0.6, 0.7, 0.8, 0.9, 1.0]
  min_child_weight          [1, 3, 5, 7]
  reg_lambda                [0.1, 1.0, 5.0, 10.0]

Total possible combinations: 38,400
We will randomly sample:     20
Time saved:                  ~1920x faster than GridSearch


In [5]:
# Base XGBoost model
base_xgb = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)

# AUC scorer
auc_scorer = make_scorer(roc_auc_score, needs_proba=True)

# RandomizedSearchCV
# cv=3 means 3-fold cross validation on each parameter set
# n_iter=20 means test 20 random parameter combinations
# verbose=2 shows progress
search = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_distributions,
    n_iter=20,
    scoring=auc_scorer,
    cv=3,
    random_state=42,
    verbose=2,
    n_jobs=1      # keep at 1 since XGBoost already uses all cores
)

print("Starting hyperparameter search...")
print("This will take 20-40 minutes on your machine.")
print("Each of the 20 combinations runs 3-fold CV on 1M rows.")
print("Go make a cup of tea!\n")

search.fit(X_train, y_train)

print("\nSearch complete!")
print(f"Best AUC (cross-validated): {search.best_score_:.4f}")
print(f"\nBest parameters found:")
for param, value in search.best_params_.items():
    print(f"  {param:<25} {value}")

Starting hyperparameter search...
This will take 20-40 minutes on your machine.
Each of the 20 combinations runs 3-fold CV on 1M rows.
Go make a cup of tea!

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=4, min_child_weight=3, n_estimators=300, reg_lambda=10.0, subsample=0.6; total time= 1.0min
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=4, min_child_weight=3, n_estimators=300, reg_lambda=10.0, subsample=0.6; total time=  58.7s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=4, min_child_weight=3, n_estimators=300, reg_lambda=10.0, subsample=0.6; total time=  57.5s
[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=5, min_child_weight=5, n_estimators=500, reg_lambda=0.1, subsample=0.6; total time= 1.7min
[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=5, min_child_weight=5, n_estimators=500, reg_lambda=0.1, subsample=0.6; total time= 1.7min
[CV] END colsample

In [16]:
# Train the best model on the full training set
print("Training final tuned model on full training set...")

tuned_model = xgb.XGBClassifier(
    **search.best_params_,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)

tuned_model.fit(X_train, y_train)

# Evaluate on held-out test set
tuned_proba = tuned_model.predict_proba(X_test)[:, 1]
tuned_auc = roc_auc_score(y_test, tuned_proba)
tuned_gini = 2 * tuned_auc - 1

from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_test, tuned_proba)
tuned_ks = max(tpr - fpr)

print(f"\n{'='*50}")
print(f"  TUNED XGBOOST — FINAL RESULTS")
print(f"{'='*50}")
print(f"  AUC:              {tuned_auc:.4f}")
print(f"  Gini Coefficient: {tuned_gini:.4f}")
print(f"  KS Statistic:     {tuned_ks:.4f}")
print(f"\n  Improvement over untuned XGBoost v2:")
print(f"  Gini: {0.4507:.4f} → {tuned_gini:.4f} "
      f"({(tuned_gini-0.4507)*100:+.2f} points)")

Training final tuned model on full training set...

  TUNED XGBOOST — FINAL RESULTS
  AUC:              0.7294
  Gini Coefficient: 0.4588
  KS Statistic:     0.3304

  Improvement over untuned XGBoost v2:
  Gini: 0.4507 → 0.4588 (+0.81 points)


In [18]:
import joblib
import os

models_dir = r"C:\Users\DELL G3\Desktop\RMIT SEM - 4\Loan Club Project\models"
os.makedirs(models_dir, exist_ok=True)

# Save the model
model_path = os.path.join(models_dir, "xgboost_credit_risk_v2.pkl")
joblib.dump(tuned_model, model_path)

# Save the feature list - important for the API later
feature_path = os.path.join(models_dir, "feature_list.pkl")
joblib.dump(list(X_train.columns), feature_path)

print(f"Model saved:    {model_path}")
print(f"Features saved: {feature_path}")
print(f"\nThese files will be loaded by the API in Phase 4")

Model saved:    C:\Users\DELL G3\Desktop\RMIT SEM - 4\Loan Club Project\models\xgboost_credit_risk_v2.pkl
Features saved: C:\Users\DELL G3\Desktop\RMIT SEM - 4\Loan Club Project\models\feature_list.pkl

These files will be loaded by the API in Phase 4


In [1]:
import xgboost as xgb
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

print(f"XGBoost version: {xgb.__version__}")

# Reload data and retrain with same parameters
data_path = r"C:\Users\DELL G3\Desktop\RMIT SEM - 4\Loan Club Project\outputs\loans_cleaned.parquet"
df = pd.read_parquet(data_path)

X = df.drop(columns=['target', 'grade', 'int_rate'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Same best parameters from tuning
tuned_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.1,
    subsample=1.0,
    colsample_bytree=1.0,
    min_child_weight=1,
    reg_lambda=5.0,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)

print("Retraining with XGBoost 2.1.1...")
tuned_model.fit(X_train, y_train)

# Overwrite saved model with 2.1.1 compatible version
model_path = r"C:\Users\DELL G3\Desktop\RMIT SEM - 4\Loan Club Project\models\xgboost_credit_risk_v2.pkl"
joblib.dump(tuned_model, model_path)

# Quick performance check
from sklearn.metrics import roc_auc_score
probas = tuned_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, probas)
gini = 2 * auc - 1

print(f"\nRetrained model performance:")
print(f"  AUC:  {auc:.4f}")
print(f"  Gini: {gini:.4f}")
print(f"\nModel saved — ready for SHAP")

XGBoost version: 2.1.1
Retraining with XGBoost 2.1.1...

Retrained model performance:
  AUC:  0.7294
  Gini: 0.4588

Model saved — ready for SHAP
